# TSLA Data Import

Imports TSLA OHLCV data from Yahoo Finance for two timeframes:
- **Low timeframe (LTF):** 1-hour bars
- **High timeframe (HTF):** 1-day bars

**Requested date range:** 2021-01-04 to 2024-09-30

> **Important:** Yahoo Finance only retains ~730 days of **hourly** data.  
> Daily data covers the full requested range, but hourly data will be limited to the last ~2 years.  
> The actual date range is reflected in the saved CSV filenames.

In [24]:
# ---------------------------------------------------------------------------
# 0. Dependencies
# ---------------------------------------------------------------------------
# Uncomment to install yfinance if not available
# !pip install yfinance --quiet

In [25]:
# ---------------------------------------------------------------------------
# 1. Imports & parameters
# ---------------------------------------------------------------------------
import yfinance as yf
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

TICKER     = "TSLA"
START_DATE = date(2021, 1, 4)
END_DATE   = date(2024, 9, 30)

# Output directory (created automatically if missing)
OUT_DIR = Path("data/raw")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ticker     : {TICKER}")
print(f"Start date : {START_DATE}")
print(f"End date   : {END_DATE}")
print(f"Output dir : {OUT_DIR.resolve()}")

Ticker     : TSLA
Start date : 2021-01-04
End date   : 2024-09-30
Output dir : C:\Users\rickc\OneDrive\文件\CUHK\RMS6007 Risk and Finance\Group Porject\RMSC6007_Liquidity-Zones\data\raw


In [26]:
# ---------------------------------------------------------------------------
# 2. Helper: chunked hourly download
# ---------------------------------------------------------------------------
# Yahoo Finance only retains ~730 days of hourly data.
# If the requested start is older the function clips it and warns.
# Each chunk is at most `chunk_days` calendar days.

def download_hourly_chunked(
    ticker: str,
    start: date,
    end: date,
    chunk_days: int = 700,
) -> pd.DataFrame:
    # Yahoo Finance hard limit: hourly data only for the last 730 days
    earliest_available = date.today() - timedelta(days=729)

    if start < earliest_available:
        print(
            f"[Warning] Hourly data is only available from {earliest_available} "
            f"(requested {start}). Adjusting start date."
        )
        start = earliest_available

    if start > end:
        raise ValueError(
            f"No hourly data available for the requested range ({start} – {end}). "
            f"Yahoo Finance retains at most 730 days of hourly history; "
            f"the earliest available date today is {earliest_available}."
        )

    frames = []
    chunk_start = start

    while chunk_start < end:
        chunk_end = min(chunk_start + timedelta(days=chunk_days), end)
        print(f"  Fetching 1h  {chunk_start} -> {chunk_end} ...", end=" ")

        df = yf.download(
            tickers=ticker,
            start=chunk_start.isoformat(),
            end=(chunk_end + timedelta(days=1)).isoformat(),  # end is exclusive in yf
            interval="1h",
            auto_adjust=True,
            progress=False,
            multi_level_index=False,
        )

        print(f"{len(df)} rows")
        if not df.empty:
            frames.append(df)
        chunk_start = chunk_end + timedelta(days=1)

    if not frames:
        raise ValueError(
            "All chunks returned empty data. "
            "Verify that the date range overlaps with the last 730 days."
        )

    combined = pd.concat(frames)
    combined = combined[~combined.index.duplicated(keep="first")]
    combined.sort_index(inplace=True)
    return combined

In [27]:
# ---------------------------------------------------------------------------
# 3. Download daily data (HTF)
# ---------------------------------------------------------------------------
print("Downloading daily (HTF) data...")

df_daily = yf.download(
    tickers=TICKER,
    start=START_DATE.isoformat(),
    end=(END_DATE + timedelta(days=1)).isoformat(),
    interval="1d",
    auto_adjust=True,
    progress=False,
    multi_level_index=False,
)

df_daily.index.name = "Datetime"
print(f"Daily rows : {len(df_daily)}")
print(f"Date range : {df_daily.index[0].date()} -> {df_daily.index[-1].date()}")
df_daily.tail(3)

Daily rows : 941
Date range : 2021-01-04 -> 2024-09-30


,Close,High,Low,Open,Volume
Datetime,,,,,
2024-09-26,254.220001,261.750000,251.529999,260.600006,67142200
2024-09-27,260.459991,260.700012,254.119995,257.380005,70988100
2024-09-30,261.630005,264.859985,255.770004,259.040009,80705700


In [33]:
# ---------------------------------------------------------------------------
# 4. Read hourly data (LTF) from local CSV
# ---------------------------------------------------------------------------
CSV_PATH = Path("data/raw/TSLA_60min.csv")

print(f"Reading hourly (LTF) data from {CSV_PATH.resolve()} ...")

df_hourly = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df_hourly.index = pd.to_datetime(df_hourly.index, dayfirst=True)
df_hourly.index.name = "Datetime"
df_hourly.sort_index(inplace=True)

# Filter to the same date range as df_daily
df_hourly = df_hourly.loc[START_DATE.isoformat() : END_DATE.isoformat()]

if df_hourly.empty:
    raise RuntimeError(f"df_hourly is empty. Check that {CSV_PATH} exists and covers {START_DATE} – {END_DATE}.")

print(f"\nHourly rows : {len(df_hourly)}")
print(f"Date range  : {df_hourly.index.min()} -> {df_hourly.index.max()}")
df_hourly.head(5)


Reading hourly (LTF) data from C:\Users\rickc\OneDrive\文件\CUHK\RMS6007 Risk and Finance\Group Porject\RMSC6007_Liquidity-Zones\data\raw\TSLA_60min.csv ...

Hourly rows : 6560
Date range  : 2021-01-04 14:00:00 -> 2024-09-30 20:00:00


,open,high,low,close,% change
Datetime,,,,,
2021-01-04 14:00:00,242.61,247.87,241.85,246.03,1.409670
2021-01-04 15:00:00,246.20,248.08,243.50,243.73,-1.003249
2021-01-04 16:00:00,243.73,245.58,242.75,243.60,-0.053338
2021-01-04 17:00:00,243.60,244.16,239.34,241.32,-0.935961
2021-01-04 18:00:00,241.32,245.48,241.10,244.38,1.268026


In [ ]:
# ---------------------------------------------------------------------------
# 5. Save to CSV
# ---------------------------------------------------------------------------
# Use actual date ranges from the downloaded data (especially important for hourly)
daily_start  = df_daily.index.min().date()
daily_end    = df_daily.index.max().date()
hourly_start = df_hourly.index.min().date()
hourly_end   = df_hourly.index.max().date()

daily_path  = OUT_DIR / f"{TICKER}_daily_{daily_start}_{daily_end}.csv"
hourly_path = OUT_DIR / f"{TICKER}_hourly_{hourly_start}_{hourly_end}.csv"

df_daily.to_csv(daily_path)
df_hourly.to_csv(hourly_path)

print(f"Saved daily  -> {daily_path}")
print(f"Saved hourly -> {hourly_path}")

Saved daily  -> data\raw\TSLA_daily_2021-01-04_2024-09-30.csv
Saved hourly -> data\raw\TSLA_hourly_2010-06-29_2025-03-07.csv


In [ ]:
# ---------------------------------------------------------------------------
# 6. Sanity check — reload and display shapes
# ---------------------------------------------------------------------------
reload_daily  = pd.read_csv(daily_path,  index_col="Datetime", parse_dates=True)
reload_hourly = pd.read_csv(hourly_path, index_col="Datetime", parse_dates=True)

print("=" * 45)
print(f"Daily  CSV shape  : {reload_daily.shape}")
print(f"Hourly CSV shape  : {reload_hourly.shape}")
print("=" * 45)
print("Daily columns : ",  reload_daily.columns.tolist())
print("Hourly columns: ",  reload_hourly.columns.tolist())
print("Daily head:")
display(reload_daily.head(3))
print("Hourly head:")
display(reload_hourly.head(3))

Daily  CSV shape  : (941, 5)
Hourly CSV shape  : (990, 5)
Daily columns :  ['Close', 'High', 'Low', 'Open', 'Volume']
Hourly columns:  ['Close', 'High', 'Low', 'Open', 'Volume']
Daily head:


,Close,High,Low,Open,Volume
Datetime,,,,,
2021-01-04,243.256668,248.163330,239.063339,239.820007,145914600
2021-01-05,245.036667,246.946671,239.733337,241.220001,96735600
2021-01-06,251.993332,258.000000,249.699997,252.830002,134100000


Hourly head:


,Close,High,Low,Open,Volume
Datetime,,,,,
2024-03-08 14:30:00+00:00,179.014999,182.729996,177.699997,181.500000,25953205
2024-03-08 15:30:00+00:00,175.059998,179.229996,174.770004,179.035004,18238810
2024-03-08 16:30:00+00:00,176.250000,176.819901,174.699997,175.070007,10327639
